# Example Comparing Schedule P Triangles across Valuation Years

In [1]:
import numpy as np
import pandas as pd

In [2]:
def compare_triangles(prior_path, curr_path, threshold=0.001, show_triangle=True):
    prior_triangle = pd.read_csv(prior_path, sep='\t')
    curr_triangle = pd.read_csv(curr_path, sep='\t')
    
    assert list(prior_triangle.columns) == list(curr_triangle.columns), 'Columns are not the same between the two tables!'
    
    joined_triangle = curr_triangle.merge(prior_triangle, how='left', on=['accident_year'], suffixes=('_curr', '_prior'))
    
    for col in joined_triangle.columns:
        if col.endswith('_curr'):
            joined_triangle[col[:-5] + '_diff'] = joined_triangle[col] - joined_triangle[col[:-5] + '_prior']
            
    joined_triangle = joined_triangle[['accident_year'] + [col for col in joined_triangle.columns if col.endswith('_diff')]]
    joined_triangle = joined_triangle[joined_triangle['accident_year'] != 'Prior']
    
    if show_triangle:
        display(joined_triangle)
        
    joined_triangle = joined_triangle.drop(columns=['accident_year'])
    
    # Check for threshold violations and collect details
    max_change = np.max(abs(joined_triangle))
    
    if max_change > threshold:
        return {
            'status': '❌Fail',
            'threshold': threshold,
            'max_change': max_change,
            'details': f'There is a change greater than {threshold}. Largest change is {max_change}'
        }
    else:
        return {
            'status': '✅Pass',
            'threshold': threshold,
            'max_change': max_change,
            'details': '-'
        }

def generate_comparison_report(output_file='validation_report.md'):
    results = []
    
    for triangle_type in ('ho_inc_loss', 'ho_paid_loss', 'ho_cwp', 'ho_open', 'ho_claims_rept'):
        for curr_year in (2020, 2021):
            try:
                result = compare_triangles(
                    f'ScheduleP/{triangle_type}_{curr_year-1}.tsv', 
                    f'ScheduleP/{triangle_type}_{curr_year}.tsv', 
                    threshold=2, 
                    show_triangle=False
                )
                
                results.append({
                    'Triangle Type': triangle_type,
                    'Year': curr_year,
                    'Status': result['status'],
                    'Threshold': result['threshold'],
                    'Max Change': f"{result['max_change']:.3f}",
                    'Details': result['details']
                })
                
            except FileNotFoundError as e:
                results.append({
                    'Triangle Type': triangle_type,
                    'Year': curr_year,
                    'Status': '❌Error',
                    'Threshold': '-',
                    'Max Change': '-',
                    'Details': f"File not found: {e.filename}"
                })
            except Exception as e:
                results.append({
                    'Triangle Type': triangle_type,
                    'Year': curr_year,
                    'Status': '❌Error',
                    'Threshold': '-',
                    'Max Change': '-',
                    'Details': f"Unexpected error: {str(e)}"
                })

    # --- Generate Markdown Report ---
    
    # Calculate Summary Stats
    total_tests = len(results)
    passed_count = sum(1 for r in results if r['Status'] == '✅Pass')
    failed_count = sum(1 for r in results if r['Status'] == '❌Fail')
    error_count = sum(1 for r in results if r['Status'] == 'Error')

    md_report = f"# Triangle Comparison Report\n\n"
    md_report += f"**Summary:** {passed_count} Passed, {failed_count} Failed, {error_count} Errors out of {total_tests} tests.\n\n"
    
    # Create Table Header
    md_report += "| Triangle Type | Year | Status | Threshold | Max Change | Details |\n"
    md_report += "|---|---|---|---|---|---|\n"
    
    # Populate Rows
    for r in results:
        status_markdown = f"**{r['Status']}**" if r['Status'] != '✅Pass' else r['Status']
        threshold_display = f"{float(r['Threshold']):.3f}" if r['Threshold'] != '-' else r['Threshold']
        max_change_display = r['Max Change'] if r['Max Change'] != '-' else r['Max Change']
        
        md_report += f"| {r['Triangle Type']} | {r['Year']} | {status_markdown} | {threshold_display} | {max_change_display} | {r['Details']} |\n"

    # Save to file and print
    with open(output_file, 'w') as f:
        f.write(md_report)
        
    print(f"Report generated successfully!")
    print(f"Saved to: {output_file}")
    print("\n--- Preview ---")
    print(md_report)

generate_comparison_report()


Report generated successfully!
Saved to: validation_report.md

--- Preview ---
# Triangle Comparison Report

**Summary:** 9 Passed, 1 Failed, 0 Errors out of 10 tests.

| Triangle Type | Year | Status | Threshold | Max Change | Details |
|---|---|---|---|---|---|
| ho_inc_loss | 2020 | ✅Pass | 2.000 | 2.000 | - |
| ho_inc_loss | 2021 | **❌Fail** | 2.000 | 6.000 | There is a change greater than 2. Largest change is 6.0 |
| ho_paid_loss | 2020 | ✅Pass | 2.000 | 2.000 | - |
| ho_paid_loss | 2021 | ✅Pass | 2.000 | 2.000 | - |
| ho_cwp | 2020 | ✅Pass | 2.000 | 2.000 | - |
| ho_cwp | 2021 | ✅Pass | 2.000 | 2.000 | - |
| ho_open | 2020 | ✅Pass | 2.000 | 1.000 | - |
| ho_open | 2021 | ✅Pass | 2.000 | 1.000 | - |
| ho_claims_rept | 2020 | ✅Pass | 2.000 | 1.000 | - |
| ho_claims_rept | 2021 | ✅Pass | 2.000 | 2.000 | - |

